# Real Estate Price Model — Simple Comparison

A clean notebook version of the simplified model-testing script.

### Workflow
1. Load the processed dataset
2. Remove flagged and systematic outliers
3. Group rare property types
4. Frequency-encode location
5. Train on log-transformed prices
6. Compare Random Forest and HistGradientBoosting
7. Cross-validate the best model


## 1. Imports and settings

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42


## 2. Find and load the dataset

This helper searches common project folders for `data.xlsx`.


In [ ]:
def find_data_file(filename="data.xlsx"):
    here = os.getcwd()

    candidates = [
        filename,
        os.path.join(here, filename),
        os.path.join(here, "..", filename),
        os.path.join(here, "..", "data", filename),
        os.path.join(here, "..", "data", "processed", filename),
        os.path.join(here, "data", filename),
        os.path.join(here, "data", "processed", filename),
    ]

    for path in candidates:
        if os.path.exists(path):
            return os.path.abspath(path)

    matches = glob.glob(
        os.path.join(here, "..", "..", "**", filename),
        recursive=True,
    )

    if matches:
        return matches[0]

    raise FileNotFoundError(
        f"Couldn't find {filename} — set the path manually."
    )

DATA_PATH = find_data_file()

print("Loading data from:", DATA_PATH)

df = pd.read_excel(DATA_PATH)

print("Raw shape:", df.shape)

df.head()


## 3. Clean the data

We:
- remove rows already marked as suspected outliers
- trim the lowest and highest 1% of `Price_per_sqft` **within each property type**
- group property types with fewer than 30 rows into `Other_rare`
- frequency-encode detailed locations
- create a log-transformed target


In [ ]:
# Remove already flagged outliers
df = df[df["Suspected_outlier"] == False].copy()

# Per-property-type trimming using Price_per_sqft
lo = df.groupby("Type")["Price_per_sqft"].transform(
    lambda s: s.quantile(0.01)
)

hi = df.groupby("Type")["Price_per_sqft"].transform(
    lambda s: s.quantile(0.99)
)

df = df[
    (df["Price_per_sqft"] >= lo)
    & (df["Price_per_sqft"] <= hi)
].copy()

print("Shape after outlier trimming:", df.shape)


In [ ]:
# Group rare property types
type_counts = df["Type"].value_counts()
rare_types = type_counts[type_counts < 30].index

df["Type_grouped"] = df["Type"].where(
    ~df["Type"].isin(rare_types),
    "Other_rare"
)

# Frequency-encode detailed location
df["Location_freq"] = df["Location"].map(
    df["Location"].value_counts()
)

# Log-transform price target
df["y_log"] = np.log1p(df["Price_pkr"])

df["Type_grouped"].value_counts()


## 4. Prepare features and train/test split

In [ ]:
NUM = [
    "Area_sqft",
    "Bedroom",
    "Bath",
    "Log_area_sqft",
    "Area_per_bedroom",
    "Bath_bedroom_ratio",
    "Location_freq",
]

CAT = [
    "Type_grouped",
    "Location_city",
]

X = df[NUM + CAT]
y_log = df["y_log"]
y_raw = df["Price_pkr"]

X_train, X_test, ylog_train, ylog_test, yraw_train, yraw_test = train_test_split(
    X,
    y_log,
    y_raw,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 5. Preprocessing

In [ ]:
preprocess = ColumnTransformer([
    ("num", "passthrough", NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
])


## 6. Train and compare models

Both models use the same preprocessing and log-price target.

The results are reported back in normal PKR as well as log-space R².


In [ ]:
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=400,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        random_state=RANDOM_STATE,
    ),
}


In [ ]:
results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocess),
        ("model", model),
    ])

    pipe.fit(X_train, ylog_train)

    pred_log = pipe.predict(X_test)
    pred_raw = np.expm1(pred_log)

    mae = mean_absolute_error(yraw_test, pred_raw)
    rmse = np.sqrt(mean_squared_error(yraw_test, pred_raw))
    r2 = r2_score(yraw_test, pred_raw)
    log_r2 = r2_score(ylog_test, pred_log)

    results.append({
        "Model": name,
        "MAE_PKR": mae,
        "RMSE_PKR": rmse,
        "R2": r2,
        "Log_R2": log_r2,
    })

    fitted_models[name] = pipe

results_df = (
    pd.DataFrame(results)
    .sort_values("Log_R2", ascending=False)
    .reset_index(drop=True)
)

results_df


## 7. Cross-validate the best model

A single train/test split can be lucky.

This uses shuffled 5-fold cross-validation to check whether the winning model performs consistently across different subsets of the dataset.


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_pipe = fitted_models[best_name]

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scores = cross_validate(
    best_pipe,
    X,
    y_log,
    cv=cv,
    scoring="r2",
    n_jobs=-1,
)

cv_mean = scores["test_score"].mean()
cv_std = scores["test_score"].std()

print("Best model:", best_name)
print("5-fold CV Log R² scores:", scores["test_score"])
print(f"Mean Log R²: {cv_mean:.3f}")
print(f"Standard deviation: {cv_std:.3f}")


## 8. Final result

Use the table above for the single 80/20 test split and the cross-validation section to judge stability.

For the hackathon, the stronger model should ideally have:
- lower MAE and RMSE
- higher R² / Log R²
- a good cross-validation average
- a reasonably small cross-validation standard deviation
